# E1 Measurement of magnitudes

## Introduction

We will cover the basics of measurement of typical magnitudes important in the performance of an axial fan. An example of performance measurement
of a centrifugal fan is revised. 

## Learning Objectives

- Learn the basics about pressure measurement
- Learn the basics about temperature measurement
- Learn the basics about velocity and flow rate measurement
- Learn the basics about torque measurement
- Learn the basics about rotational velocity measurement
- Perform flow rate, head, power consumption and efficiency computation from raw measurements of basic magnitudes in the flow and the machine 

## Previous tasks (about 2 hours)

For the book by Dick

Dick, Erik. Fundamentals of Turbomachines. Vol. 109. Dordrecht, The Netherlands: Springer, 2015.

which is [available in the UPC library](https://link-springer-com.recursos.biblioteca.upc.edu/book/10.1007/978-3-030-93578-8)


 - Read sections 5.1-5.5

## Some simple questions

For the typical value of pressure rise in an axial fan, what is the most suitable type of pressure measurement equipment? If it has to be calibrated with a U tube with water, what is the typical height difference expected?

Answer

***

What kind of flow measurement device is more convenient for air flow rate in an axial fan? What does it mean that the flow has to be
_fully developed_ and why is that so important?

Answer

***

What kind of torque measurement device would you use for an axial fan and why?

Answer

***

What kind of rotational velocity  measurement device would you use for an axial fan and why?

Answer

***

## Tasks

This task is adapted from section 5.7 in the book by Dick

The objective is to draw the **performance curve** ($Y = \frac{\Delta p_0}{\rho}$, $P$ and $\eta$ versus $Q$) for a given rotational velocity of a **centrifugal fan**.

The test rig is sketched in this figure

![test_rig](images/E1_test_rig_fan.png)

The rotational velocity of the driving motor of the fan is not constant, but function of the consumed power. The torque produced by the motor and the
rotational velocity can be approximated as a **linear function** of the consumed power. It is measured that for a torque of $M = 0 \,\text{N\,m}$ electric power is
$P_e = 165\,\text{W}$ and for $M = 12\,\text{N\,m}$ it is $P_e = 4225\,\text{W}$. Likewise, for a rotational velocity $N = 2750\,\text{rpm}$ consumed power is 
$P_e = 400\,\text{W}$ and for $N = 2630\,\text{rpm}$ it is $P_e = 3620\,\text{W}$. Ambient pressure and temperatures are $P_a = 100.4\,\text{kPa}$ and $T_a = 21.0 \, \text{°C}$.

The nozzle flow meter is a standard design type ISA 1932, according to [ISO-5167](https://plataforma-aenormas-aenor-com.recursos.biblioteca.upc.edu/standard/UNE/N0071852), with $D = 182.9\,\text{mm}$ and $d = 114.0\,\text{mm}$.


For a given cone control position, consumed power ($P_e$) in $\text{W}$, temperature ($T$) in Celsius degrees, pressure ($p$) at fan outlet, pressures at nozzle inlet ($p_1$) and pressure difference
in nozzle ($\Delta p = p_2 - p_1$) in $\text{kPa}$ are measured. The data are that stored in pandas data frame `exp_data`

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', '{:.4g}'.format) # To limit to 4 the number of significant digits in the output

P_e = np.array([1200, 1480, 1840, 2240, 2640, 3000, 3400, 3760])
T = np.array([27.4, 25.7, 24.6, 24.0, 23.7, 23.6, 23.2, 22.5])
p = np.array([2.92, 3.06, 3.13, 3.13, 3.08, 2.92, 2.60, 2.23])
p1 = np.array([2.92, 3.04, 3.09, 3.06, 2.96, 2.76, 2.38, 1.95])
Delta_p = np.array([0.0, 0.07, 0.27, 0.60, 1.05, 1.60, 2.38, 3.25])
exp_data = pd.DataFrame({
    'P_e (W)': P_e,
    'T (°C)': T,
    'p (kPa)': p,
    'p1 (kPa)': p1,
    'Delta_p (kPa)': Delta_p
})
exp_data # to display the experimental data in a tabular format

1. Compute actual rotational velocity in $\text{rpm}$ and torque in $\text{N\,m}$ from above lineal correlation. It can be done directly in the pandas dataframe with an expression of the type `exp_data['N (rpm)'] = (N2-N1)/(Pe_2_N-Pe_1_N)*(exp_data['P_e (W)'] -Pe_1_N) + N1`.

In [ ]:
N2 = 2630
N1 = 2750
Pe_2_N = 3620
Pe_1_N = 400
exp_data['N (rpm)'] = (N2-N1) * (exp_data['P_e (W)'] - Pe_1_N) / (Pe_2_N - Pe_1_N) + N1
M2 = 12
M1 = 0
Pe_2_M = 4225
Pe_1_M = 165
exp_data['M (N m)'] = (M2-M1) * (exp_data['P_e (W)'] - Pe_1_M) / (Pe_2_M - Pe_1_M) + M1
exp_data

2. We are going to need density of gas in the flow meter nozzle in order to compute mass flow rate, at point 1. It can be done by using ideal gas equation, and assuming that $T_1 \approx T$. Create a column in the dataframe with the density at point 1, $\rho_1 (\text{kg/m}^3)$,
$$
\rho_1 = \frac{p_1}{R T}
$$
where $ R = 287.05\,\text{J/(kg\,K)}$ and $p$ and $T$ are **absolute** pressure and temperature.

Define also ambient pressure, temperature and density, that may be used later.

1. Also viscosity will be nedded to calculate Reynolds number. Define a new column with viscosity of fluid at point 1, that can be related with temperture with the expression
   $$
    \mu_1(T_1) = (17.1 + 0.048\, T_1)\times 10^{-6} \, \text{Pa s}
   $$
   where $T_1$ is expressed in Celsius degrees for $-20 °C < T_1 < 100 °C$. This is the linear correlation recommended by the [ISO 5801:2019](https://plataforma-aenormas-aenor-com.recursos.biblioteca.upc.edu/standard/UNE/N0061895).

4. Compute mass flow rate for each operation point according to ISO-5167 procedure. Since the discharge coefficient is function of the Reynolds number, which is
function of flow rate, it has to be done by iterations until convergence is reached. This can be directly done using the 
[`flow_meter`](https://fluids.readthedocs.io/fluids.flow_meter.html) module in the `fluid` 
python package

``` python
from fluids.flow_meter import *

differential_pressure_meter_solver(D=0.1829, 
                                   D2=0.114,
                                   rho=1.2,
                                   mu=1.8e-5,
                                   P1 = 103000,
                                   P2 = 102500,
                                   k = 1.4,
                                   meter_type='ISA 1932 nozzle'
)

0.3654245417430694
```

Note that this module does not accept `Series` or `arrays` as inputs (it is not vectorized), so a loop has to be coded. Also note that pressures have to be passed 
as absolute values.


5. Compute density of air at the outlet of the fan and use mass flow rate to calculate volumetric flow rate in the fan. We need it to get velocity
   and total pressure.

6. Compute average velocity of air at fan outlet with flow rate and duct section

7. Estimate mechanical energy rise $Y = \frac{\Delta p_0}{\rho}$ in $\text{J/kg}$, taking $\rho$ as the average between the inlet and outlet densities.

8. Estimate mechanical efficiency as 
$$
\eta = \frac{\Delta p_0 Q}{\omega M}
$$

9. Scale flow rate, mechanical energy and consumed mechanical power to the idling rotational velocity $N_0 = 2750 \,\text{rpm}$, considering similitude relationships 
(see [Notebook D1](../1_Design/D1_fundamentals_and_dimensional_analysis.ipynb))
$$
Q \sim N \\
Y \sim N^2 \\
P \sim N ^3 \\
\eta \sim \text{constant}
$$ 

10. Plot performance curves $Y(Q)$, $P/Q)$ and $\eta(Q)$ and draw conclusions Use `matplotlib` with a code similar to this:
```python
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

exp_data.plot(x='Q_N (m^3/s)', y='Y_N (J/kg)', kind='line', marker='o', legend=False, ax=axes[0])
exp_data.plot(x='Q_N (m^3/s)', y='P_N (kW)', kind='line', marker='o', legend=False, ax=axes[1])
exp_data.plot(x='Q_N (m^3/s)', y='eta', kind='line', marker='o', legend=False, ax=axes[2])
axes[0].set_title('Fan performance curves for N = 2750 rpm')
axes[0].set_ylabel(r'$Y\,\text{(J/kg)}$')
axes[1].set_ylabel(r'$P\,\text{(kW)}$')
axes[2].set_xlabel(r'$Q\,\text{(m}^3\text{/s)}$')
axes[2].set_ylabel(r'$\eta$')
axes[2].set_ylim(0, 1)
for ax in axes:
    ax.grid()
plt.tight_layout()
```

## Your project

With respect to your project, consider the following points:
- According to the nominal flow rate of your axial fan, which flow rate device would be more convenient? In this example a ISA 1932 nozzle is used, but for larger flow rates, lower pressure, a different design could be more convenient. Perhaps a long-radius nozzle or a bellmouth inlet nozzle.
- Browse the catalogue by [Wika](https://www.wika.com/es-es/pagina_inicial.WIKA) or [Keller](https://keller-pressure.com) the suitable digital manometers for your application
- Look for the appropriate thermocouple for the measurement of temperature of air. See for instance [TC Direct](https://www.tcdirect.es/) or any other manufacturer/seller. Consider also how are you going to register the measurement. 